# Lateral-drag form-factor workflow

This notebook builds coastline-only, GIB-only, and combined CICE lateral-drag form-factor files using `shuga.grid.lateral_drag.FormFactors`, then plots regional diagnostics using PyGMT.

The notebook is designed for Gadi paths and the current Antarctic v7.9 coastline / Kaihong Jiao v1.2 grounded-iceberg workflow. Adjust paths and overwrite flags before running.

In [1]:
from __future__ import annotations

import os
import sys
import warnings
from pathlib import Path
from typing import Mapping, Sequence

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
from IPython.display import Image, display

xr.set_options(keep_attrs=True)
warnings.filterwarnings('default')

repo_root = Path.home() / 'AFIM' / 'src' / 'mawsons-chest'
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

In [2]:
from shuga.core.types import RunSpec, ClassificationSpec, LateralDragSpec
from shuga.core.paths import ShugaPaths
from shuga.grid.lateral_drag import FormFactors
from shuga import regions

## Configuration

Set the form-factor input/output paths and instantiate the `FormFactors` builder.

In [ ]:
PROJECT    = 'gv90'
USER       = 'da1339'
HEMISPHERE = 'SH'
ICE_TYPE   = 'FI'
GRID_TYPE  = 'Tc'

ISPD_THRESH  = 5.0e-4
BIN_WINDOW   = 11
BIN_MIN_DAYS = 9
ROLL_WINDOW  = 15

OVERWRITE_COAST    = False
OVERWRITE_GIB      = False
OVERWRITE_COMBINED = True

D_root = Path('/g/data/gv90/da1339')
D_FF   = D_root / 'form_factors'
D_cst  = D_root / 'coastlines' / 'high_res_coast'
D_GIB  = D_root / 'grounded_icebergs' / 'Kaihong_Jiao'
D_FF.mkdir(parents=True, exist_ok=True)

P_Hres_cst = D_cst / 'add_coastline_high_res_polygon_v7_9.shp'

GIB_SPECS = {
    'v0p9': D_GIB / 'Antarctic_Grounded_Iceberg_Dataset_Sentinel1_v0.9.gpkg',
    'v1p0': D_GIB / 'Antarctic_Grounded_Iceberg_Dataset_Sentinel1_v1.0.gpkg',
    'v1p1': D_GIB / 'Antarctic_Grounded_Iceberg_Dataset_Sentinel1_v1.1.gpkg',
    'v1p2': D_GIB / 'Antarctic_Grounded_Iceberg_Dataset_Sentinel1_v1.2.gpkg',
}

cst_ver  = 'v7p9'
gib_ver  = 'v1p2'
cmb_meth = 'max'

P_FF_cst = D_FF / f'FF_ADD-Hres-cst-{cst_ver}_Liu_CICE.nc'
P_FF_gib = D_FF / f'FF_GIB-{gib_ver}_perimeter_Liu_CICE.nc'
P_FF_cmb = D_FF / f'FF_combined_meth-{cmb_meth}_cst-{cst_ver}-Liu_GIB-{gib_ver}-perimeter_Liu_CICE.nc'

run_cfg = RunSpec(
    sim_name   = 'LD-static-Cs1e-3',
    start_date = '1999-01-01',
    end_date   = '1999-12-31',
    hemisphere = HEMISPHERE,
    project    = PROJECT,
    user       = USER,
)

cls_cfg = ClassificationSpec(
    ice_type     = ICE_TYPE,
    grid_type    = GRID_TYPE,
    ispd_thresh  = ISPD_THRESH,
    methods      = ('binary-days',),
    bin_window   = BIN_WINDOW,
    bin_min_days = BIN_MIN_DAYS,
    roll_window  = ROLL_WINDOW,
)

pth_cfg = ShugaPaths(run_cfg=run_cfg, cls_cfg=cls_cfg)
LD_cfg  = LateralDragSpec()
FF      = FormFactors(pth_cfg=pth_cfg, LD_cfg=LD_cfg)

## Build form-factor products

These cells generate the coastline-only, GIB-only, and combined NetCDF files. For Liu-style length-density fields, keep `clip_max=None` unless you are deliberately running a bounded sensitivity experiment.

In [ ]:
ds_coast = FF.build_FF_from_Hres_coast_Liu(
    P_Hres_cst       = P_Hres_cst,
    P_out            = P_FF_cst,
    source_lat_limit = -60.0,
    overwrite        = OVERWRITE_COAST,
    clip_max         = None,
)

FF.assert_CICE_F2_file_compatibility(P_FF_cst, nx_global=1440, ny_global=1080)

In [ ]:
ds_gib = FF.build_FF_from_GIB_perimeter(
    P_GIB                 = GIB_SPECS[gib_ver],
    P_out                 = P_FF_gib,
    overwrite             = OVERWRITE_GIB,
    include_area_fraction = True,
    area_component_mode   = 'diagnostic',
    clip_max              = None,
)

FF.assert_CICE_F2_file_compatibility(P_FF_gib, nx_global=1440, ny_global=1080)

In [ ]:
ds_cmb = FF.build_FF_combined_CICE(
    P_FF_cst          = P_FF_cst,
    P_FF_GIB          = P_FF_gib,
    P_out             = P_FF_cmb,
    FF_combine_method = cmb_meth,
    overwrite         = OVERWRITE_COMBINED,
    clip_max          = None,
)

FF.assert_CICE_F2_file_compatibility(P_FF_cmb, nx_global=1440, ny_global=1080)

## Quick product summaries

In [ ]:
for path in [P_FF_cst, P_FF_gib, P_FF_cmb]:
    ds = xr.open_dataset(path)
    try:
        print('\n' + '=' * 80)
        print(path)
        print(ds)
        print('FFx max:', float(ds['FFx'].max()))
        print('FFy max:', float(ds['FFy'].max()))
        print('nonzero:', int(((ds['FFx'] > 0) | (ds['FFy'] > 0)).sum()))
        print('diagnostics:', [v for v in ds.data_vars if v not in {'FFx', 'FFy', 'lon', 'lat'}])
    finally:
        ds.close()


## Plotting helpers

The helper cell below is copied from the current PyGMT workflow: plot $|F_{2xy}|$, annotate regional diagnostics, and save one PNG per Antarctic region. If this stabilises, it should move into `shuga/plotting/lateral_drag.py`.

In [ ]:
def load_form_factor_dataset(path: Path | str) -> xr.Dataset:
    """Open a form-factor dataset eagerly so the file can be closed safely."""
    with xr.open_dataset(path) as ds:
        return ds.load()


def first_present(ds: xr.Dataset, candidates: list[str]) -> str | None:
    for name in candidates:
        if name in ds:
            return name
    return None


def form_factor_magnitude(ds: xr.Dataset) -> xr.DataArray:
    """Infer $|F_{2xy}|$ from FFx/FFy-like variables."""
    candidates_x = ["FFx", "F2x", "F2E", "F2N_x", "F2_x"]
    candidates_y = ["FFy", "F2y", "F2N", "F2E_y", "F2_y"]
    x = first_present(ds, candidates_x)
    y = first_present(ds, candidates_y)
    if x and y:
        return xr.apply_ufunc(np.hypot, ds[x], ds[y], dask="allowed").rename("FFmag")
    if x:
        return abs(ds[x]).rename("FFmag")
    raise KeyError(f"Could not infer form-factor component fields from {list(ds.data_vars)}")


def infer_lonlat_names(ds: xr.Dataset) -> tuple[str, str]:
    lon_name = first_present(ds, ["lon", "TLON", "ULON", "NLON"])
    lat_name = first_present(ds, ["lat", "TLAT", "ULAT", "NLAT"])
    if lon_name is None or lat_name is None:
        raise KeyError("Need lon/lat variables in form-factor dataset.")
    return lon_name, lat_name


def lon_to_180(lon):
    lon = np.asarray(lon, dtype="float64")
    return ((lon + 180.0) % 360.0) - 180.0


def lon_to_360(lon):
    lon = np.asarray(lon, dtype="float64")
    return lon % 360.0


def central_meridian_from_region(region):
    lon_min, lon_max, *_ = region
    if lon_min <= lon_max:
        return 0.5 * (lon_min + lon_max)
    lon_min_360 = lon_min % 360.0
    lon_max_360 = (lon_max % 360.0) + 360.0
    mc = 0.5 * (lon_min_360 + lon_max_360)
    if mc > 180.0:
        mc -= 360.0
    return mc


def region_plot_params(region_name: str, ANTARCTIC_REGIONS):
    if region_name == "SH":
        return SH_REGION, SH_PROJECTION, False, SH_REGION
    info = ANTARCTIC_REGIONS[region_name]
    plot_region = list(info["plot_region"])
    lon_min, lon_max, lat_min, lat_max = plot_region
    cross_dateline = lon_min > lon_max
    mc = central_meridian_from_region(plot_region)
    projection_template = info.get("projection", "S{MC}/-90/{fig_size}c")
    proj = projection_template.format(MC=f"{mc:g}", fig_size=f"{REGION_WIDTH_CM:g}")
    if cross_dateline:
        lon_min_360 = lon_min % 360.0
        lon_max_360 = (lon_max % 360.0) + 360.0
        plot_region = [lon_min_360, lon_max_360, lat_min, lat_max]
    return plot_region, proj, cross_dateline, [lon_min, lon_max, lat_min, lat_max]


def region_keep_mask(lon, lat, val, region_name: str, ANTARCTIC_REGIONS):
    plot_region, proj, cross_dateline, bounds = region_plot_params(region_name, ANTARCTIC_REGIONS)
    lon = np.asarray(lon)
    lat = np.asarray(lat)
    val = np.asarray(val)
    finite = np.isfinite(lon) & np.isfinite(lat) & np.isfinite(val) & (val > 0)
    if region_name == "SH":
        lon_plot = lon_to_180(lon)
        keep = finite & (lat >= SH_REGION[2]) & (lat <= SH_REGION[3])
        return keep, lon_plot, plot_region, proj
    lon_min, lon_max, lat_min, lat_max = bounds
    if cross_dateline:
        lon_plot = lon_to_360(lon)
        lon_min_360 = lon_min % 360.0
        lon_max_360 = (lon_max % 360.0) + 360.0
        lon_work = lon_plot.copy()
        lon_work[lon_work < lon_min_360] += 360.0
        keep = finite & (lon_work >= lon_min_360) & (lon_work <= lon_max_360) & (lat >= lat_min) & (lat <= lat_max)
        return keep, lon_work, plot_region, proj
    lon_plot = lon_to_180(lon)
    keep = finite & (lon_plot >= lon_min) & (lon_plot <= lon_max) & (lat >= lat_min) & (lat <= lat_max)
    return keep, lon_plot, plot_region, proj


def diagnostic_vars_for_product(product_kind: str) -> list[str]:
    product_kind = str(product_kind).lower().replace("_", "-")
    if product_kind in {"cst-only", "cst", "coast", "coast-only"}:
        return ["coast_n_source_hits"]
    if product_kind in {"gib-only", "gib", "gib-perimeter"}:
        return ["GIB_area_frac", "GIB_n_perimeter_hits"]
    if product_kind in {"cmb", "combined", "combine"}:
        return ["coast_n_source_hits", "GIB_area_frac", "GIB_n_perimeter_hits"]
    raise ValueError(f"Unknown product_kind: {product_kind}")


def _as_flat_region_samples(ds: xr.Dataset, varname: str, region_name: str, ANTARCTIC_REGIONS):
    if varname not in ds:
        return np.array([]), np.array([]), np.array([])
    lon_name, lat_name = infer_lonlat_names(ds)
    lon = ds[lon_name].values.ravel()
    lat = ds[lat_name].values.ravel()
    val = ds[varname].values.ravel()
    keep, _, _, _ = region_keep_mask(lon, lat, val, region_name, ANTARCTIC_REGIONS)
    vv = np.asarray(val[keep], dtype="float64")
    llon = lon_to_180(np.asarray(lon[keep], dtype="float64"))
    llat = np.asarray(lat[keep], dtype="float64")
    good = np.isfinite(vv) & np.isfinite(llon) & np.isfinite(llat) & (vv > 0)
    return vv[good], llon[good], llat[good]


def _fmt_lonlat(lon: float, lat: float) -> str:
    if not np.isfinite(lon) or not np.isfinite(lat):
        return "nan,nan"
    lon_hemi = "E" if lon >= 0 else "W"
    lat_hemi = "N" if lat >= 0 else "S"
    return f"{abs(lon):.2f}{lon_hemi}, {abs(lat):.2f}{lat_hemi}"


def _regional_max_with_location(vv, llon, llat):
    if vv.size == 0:
        return np.nan, np.nan, np.nan
    imax = int(np.nanargmax(vv))
    return float(vv[imax]), float(llon[imax]), float(llat[imax])


def _fmt_stat_value(x: float, varname: str) -> str:
    if not np.isfinite(x):
        return "nan"
    if varname in {"GIB_area_frac"}:
        return f"{x:.3g}"
    if "hits" in varname:
        return f"{x:.0f}"
    if abs(x) >= 1.0e4 or (0 < abs(x) < 1.0e-2):
        return f"{x:.2e}"
    return f"{x:.3g}"


def _diagnostic_single_line(ds, varname, region_name, ANTARCTIC_REGIONS, label_map, include_missing=False):
    if varname not in ds:
        return f"{label_map.get(varname, varname)}: missing" if include_missing else None
    vv, llon, llat = _as_flat_region_samples(ds, varname, region_name, ANTARCTIC_REGIONS)
    lab = label_map.get(varname, varname)
    if vv.size == 0:
        return f"{lab}: n=0"
    vmax, vmax_lon, vmax_lat = _regional_max_with_location(vv, llon, llat)
    p95 = float(np.nanpercentile(vv, 95))
    vsum = float(np.nansum(vv))
    return (f"{lab}: n={vv.size:,}, max={_fmt_stat_value(vmax, varname)} at {_fmt_lonlat(vmax_lon, vmax_lat)}, "
            f"p95={_fmt_stat_value(p95, varname)}, sum={_fmt_stat_value(vsum, varname)}")


def region_diagnostic_stats_text(ds, region_name, ANTARCTIC_REGIONS, product_kind, diagnostics=None, include_missing=False):
    if diagnostics is None:
        diagnostics = diagnostic_vars_for_product(product_kind)
    label_map = {
        "coast_n_source_hits": "CST hits",
        "GIB_area_frac": "GIB area frac",
        "GIB_n_perimeter_hits": "GIB perim hits",
    }
    raw_lines = []
    for v in list(diagnostics):
        line = _diagnostic_single_line(ds, v, region_name, ANTARCTIC_REGIONS, label_map, include_missing)
        if line is not None:
            raw_lines.append(line)
    if len(raw_lines) <= 2:
        return raw_lines
    return [raw_lines[0], " | ".join(raw_lines[1:])]


def _middle_lon_from_plot_lons(lons: Sequence[float]) -> float:
    return 0.5 * (float(lons[0]) + float(lons[1]))


def diagnostic_text_xy(region_name: str, lons: Sequence[float], lats: Sequence[float]) -> tuple[float, float]:
    if str(region_name).upper() == "SH":
        return 0.0, -90.0
    x = _middle_lon_from_plot_lons(lons)
    y = float(lats[0]) - 1.0
    return x, y


def add_region_diagnostic_text(fig, ds, region_name, ANTARCTIC_REGIONS, product_kind, diagnostics=None,
                               lons=None, lats=None, font="12p,Helvetica,black", fill="white@15",
                               pen="0.25p,black", clearance="0.08c/0.08c", justify="CM",
                               no_clip=True, line_lat_offset_deg=0.5, debug=False):
    if lons is None or lats is None:
        raise ValueError("Pass lons=plot_region[0:2], lats=plot_region[2:4].")
    lines = region_diagnostic_stats_text(ds, region_name, ANTARCTIC_REGIONS, product_kind, diagnostics)
    if not lines:
        return
    x_text, y_text = diagnostic_text_xy(region_name, lons, lats)
    if debug:
        print(f"text figure base position: {x_text},{y_text}")
        for k, line in enumerate(lines):
            print(f"diagnostic line {k + 1}: {line}")
    for k, line in enumerate(lines):
        y_line = y_text - k * float(line_lat_offset_deg)
        kwargs = dict(x=x_text, y=y_line, text=line, font=font, justify=justify, no_clip=no_clip)
        if fill is not None:
            kwargs["fill"] = fill
        if pen is not None:
            kwargs["pen"] = pen
        if clearance is not None:
            kwargs["clearance"] = clearance
        fig.text(**kwargs)


def plot_form_factor_pygmt(ds, out: Path | None = None, title="Combined lateral-drag form factor",
                           region_name="SH", ANTARCTIC_REGIONS=None, product_kind="cmb",
                           diagnostics=None, show_diagnostics=True):
    import pygmt
    if ANTARCTIC_REGIONS is None:
        ANTARCTIC_REGIONS = {}
    fmag = form_factor_magnitude(ds)
    lon_name, lat_name = infer_lonlat_names(ds)
    lon = ds[lon_name].values.ravel()
    lat = ds[lat_name].values.ravel()
    val = fmag.values.ravel()
    keep, lon_plot, plot_region, proj = region_keep_mask(lon, lat, val, region_name, ANTARCTIC_REGIONS)
    fig = pygmt.Figure()
    fig.basemap(region=plot_region, projection=proj, frame=["af", f"+t{title}"])
    pygmt.makecpt(cmap=CPT_CMAP, series=CPT_SERIES)
    fig.coast(shorelines="0.4p,black", land="lightgray", water="white")
    fig.plot(x=lon_plot[keep], y=lat[keep], style=POINT_STYLE, fill=val[keep], cmap=True, pen=None)
    fig.colorbar(frame="xaf+l@[|F_{2xy}|@[")
    if show_diagnostics:
        add_region_diagnostic_text(
            fig=fig,
            ds=ds,
            region_name=region_name,
            ANTARCTIC_REGIONS=ANTARCTIC_REGIONS,
            product_kind=product_kind,
            diagnostics=diagnostics,
            lons=plot_region[0:2],
            lats=plot_region[2:4],
            font="12p,Helvetica,black",
            fill="white@15",
            pen="0.25p,black",
            clearance="0.08c/0.08c",
            justify="CM",
        )
    if out is not None:
        out.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(str(out))
        print(f"Saved: {out}")
    fig.show()

## Generate figures

This assumes the plotting helper cell above has been executed and that the output directory mirrors the tree used in `shuga/docs/figs`.

In [ ]:
D_root_out = Path('/g/data/gv90/da1339/GRAPHICAL/LD-pub-workspace/form-factors')

SH_REGION     = [-180, 180, -90, -60]
SH_PROJECTION = 'S0/-90/20c'
REGION_WIDTH_CM = 16
CPT_CMAP   = 'cmocean/matter'
CPT_SERIES = [0, 3]
POINT_STYLE = 's0.125c'

ANTARCTIC_REGIONS = regions.ANTARCTIC_8_REGIONS
ALL_REGIONS = ['SH'] + list(ANTARCTIC_REGIONS.keys())
print(ALL_REGIONS)

In [ ]:
# CST-only
with xr.open_dataset(P_FF_cst) as ds:
    D_ = D_root_out / 'cst_only' / 'no_taper'
    for region_name in ALL_REGIONS:
        P_png   = D_ / region_name / f'FF_cst-only_cst-{cst_ver}_{region_name}.png'
        tit_str = 'CST-only no-taper-distance'
        plot_form_factor_pygmt(
            ds=ds,
            out=P_png,
            title=tit_str,
            region_name=region_name,
            ANTARCTIC_REGIONS=ANTARCTIC_REGIONS,
            product_kind='cst-only',
            diagnostics=['coast_n_source_hits'],
            show_diagnostics=True,
        )

In [ ]:
# GIB-only
with xr.open_dataset(P_FF_gib) as ds:
    D_ = D_root_out / 'gib_only' / 'no_taper'
    for region_name in ALL_REGIONS:
        P_png   = D_ / region_name / f'FF_gib-only_GIB-{gib_ver}_{region_name}.png'
        tit_str = 'GIB-only no-taper-dist perimeter'
        plot_form_factor_pygmt(
            ds=ds,
            out=P_png,
            title=tit_str,
            region_name=region_name,
            ANTARCTIC_REGIONS=ANTARCTIC_REGIONS,
            product_kind='gib-only',
            diagnostics=['GIB_area_frac', 'GIB_n_perimeter_hits'],
            show_diagnostics=True,
        )

In [ ]:
# Combined
with xr.open_dataset(P_FF_cmb) as ds:
    D_ = D_root_out / 'cmb' / 'no_taper'
    for region_name in ALL_REGIONS:
        P_png   = D_ / region_name / f'FF_cmb_meth-{cmb_meth}_cst-{cst_ver}_GIB-{gib_ver}_{region_name}.png'
        tit_str = f'CST+GIB no-taper {cmb_meth}'
        plot_form_factor_pygmt(
            ds=ds,
            out=P_png,
            title=tit_str,
            region_name=region_name,
            ANTARCTIC_REGIONS=ANTARCTIC_REGIONS,
            product_kind='cmb',
            diagnostics=['coast_n_source_hits', 'GIB_area_frac', 'GIB_n_perimeter_hits'],
            show_diagnostics=True,
        )

## CICE namelist reminder

Use the combined product as `F2_file` and keep the variables as `FFx`/`FFy`.

In [ ]:
print(f"F2_file       = '{P_FF_cmb}'")
print("F2x_varname   = 'FFx'")
print("F2y_varname   = 'FFy'")
print("F2_map_method = 'max'    ! or 'avg'; this maps T-cell source fields to E/N faces inside CICE")